# 🩸 Acute Lymphoblastic Leukemia (ALL) Dual-Stream Classifier
## Full GPU Training Notebook on Kaggle

This notebook trains the dual-stream deep learning classifier for Acute Lymphoblastic Leukemia blood-smear images:
- **Stream A**: HSV color-threshold segmented images (Giemsa-stained nuclei & cytoplasm)
- **Stream B**: Original blood-smear images
- **Two-Phase Training**:
  1. Head training with frozen backbones at `lr = 1e-3`
  2. Fine-tuning top 30 backbone layers at `lr = 1e-5`
- **Classes**: Hematogones (Benign), Early Pre-B ALL, Pre-B ALL, Pro-B ALL

### Step 1: Check GPU Accelerator

In [ ]:
!nvidia-smi

import tensorflow as tf
print("TensorFlow Version:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print("Detected GPUs     :", gpus)
assert len(gpus) > 0, "No GPU found! Enable a GPU accelerator in Kaggle Notebook Settings (T4 or P100)."

### Step 2: Clone Repository & Set Working Directory

In [ ]:
import os
from pathlib import Path

repo_dir = Path("/kaggle/working/leukemia-all-classifier")

if not repo_dir.exists():
    !git clone https://github.com/HemangDulange/Leukemia.git /kaggle/working/leukemia-all-classifier

%cd /kaggle/working/leukemia-all-classifier
!pwd


### Step 3: Install Training Dependencies

In [ ]:
!pip install -r requirements-train.txt --quiet
print("✅ Dependencies installed successfully.")

### Step 4: Link Kaggle Dataset into `data/raw`
Attach the dataset `mehradaria/leukemia` in Kaggle's **Add Input** sidebar.

In [ ]:
import shutil
from pathlib import Path

# Search candidate Kaggle input paths
possible_inputs = [
    Path("/kaggle/input/leukemia"),
    Path("/kaggle/input/datasets/mehradaria/leukemia"),
    Path("/kaggle/input/acute-lymphoblastic-leukemia-all-image-dataset"),
]

found_input = None
for p in possible_inputs:
    if p.exists():
        found_input = p
        break

print("Found input dataset at:", found_input)

# Link into project data/raw
raw_dir = Path("data/raw")
raw_dir.mkdir(parents=True, exist_ok=True)

if found_input:
    orig_src = found_input / "Original"
    seg_src = found_input / "Segmented"
    if orig_src.exists() and not (raw_dir / "Original").exists():
        os.symlink(orig_src, raw_dir / "Original", target_is_directory=True)
    if seg_src.exists() and not (raw_dir / "Segmented").exists():
        os.symlink(seg_src, raw_dir / "Segmented", target_is_directory=True)

!ls -la data/raw

### Step 5: Data Pairing & Stratified Splits
Generates stratified splits (70% train, 15% val, 15% test, seed 42) and evaluates HSV segmentation.

In [ ]:
!python scripts/prepare_data.py --force --samples 50

### Step 5b: Precompute Segmented Pairs (Zero Train/Serve Skew)
Converts full-resolution original images through unified `prepare_pair()` into lossless PNGs at `data/processed/orig` and `data/processed/seg`.

In [ ]:
!python scripts/precompute_segmented.py


### Step 6: Full Two-Phase GPU Training
- **Phase 1**: Head training with frozen backbones at `lr = 1e-3`
- **Phase 2**: Fine-tuning top 30 backbone layers at `lr = 1e-5`
- **Callbacks**: EarlyStopping, ReduceLROnPlateau, ModelCheckpoint (monitoring `val_accuracy`)

In [ ]:
!python scripts/train.py

### Step 7: Evaluate Model on Test Split

In [ ]:
!python scripts/evaluate.py

### Step 7b: Check Serving Skew (Predictor vs Dataset Pipeline)
Verifies that live inference on raw images matches offline dataset batch evaluation within <= 1.5% margin.

In [ ]:
!python scripts/check_serving_skew.py


### Step 7c: Generate Clean Evaluation Samples for App
Selects 1 correctly classified test image per subtype and bundles them into `/kaggle/working/samples.zip`.

In [ ]:
!python scripts/make_samples.py


### Step 8: Visualize Metrics & Performance Plots

In [ ]:
import json
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

# Display training history curves
hist_img_path = Path("reports/figures/training_history.png")
if hist_img_path.exists():
    plt.figure(figsize=(12, 5))
    plt.imshow(mpimg.imread(str(hist_img_path)))
    plt.axis("off")
    plt.title("Training History (Accuracy & Loss)", fontsize=14)
    plt.show()

# Display test confusion matrix
cm_img_path = Path("reports/figures/confusion_matrix.png")
if cm_img_path.exists():
    plt.figure(figsize=(8, 7))
    plt.imshow(mpimg.imread(str(cm_img_path)))
    plt.axis("off")
    plt.title("Test Set Confusion Matrix", fontsize=14)
    plt.show()

# Print metrics JSON
metrics_path = Path("reports/metrics/test_metrics.json")
if metrics_path.exists():
    with metrics_path.open() as f:
        m = json.load(f)
    print("Test Macro AUC:", m.get("auc_macro_ovr"))
    print("\nClassification Report:")
    print(json.dumps(m.get("classification_report"), indent=2))

### Step 9: Verify Final Saved Model Artifact

In [ ]:
model_path = Path("models/dual_stream.keras")
if model_path.exists():
    size_mb = model_path.stat().st_size / (1024 * 1024)
    print(f"✅ Final model successfully saved: {model_path} ({size_mb:.2f} MB)")
else:
    print("⚠️ Model file not found at expected path.")